# MambaDSR — train on Colab

Runtime → Change runtime type → **GPU** (A100/L4 recommended; Mamba3 kernels may not run on a T4).

**Colab secrets** (🔑 icon, left sidebar):
- `GITHUB_TOKEN` — the repo is private
- `KAGGLE_USERNAME`, `KAGGLE_KEY` — only if downloading raw UASpeech

**On Google Drive** (under `DRIVE_DIR`):
- `processed.zip` — output of `preprocess.ipynb`, containing `processed/libreTTS/...` and `processed/ua/...`
- `checkpoints/` — created automatically; every checkpoint lands here so it survives disconnects

In [ ]:
MODE = "pretrain"              # pretrain | finetune | e2e | vocoder_adapt
RESUME = None                  # e.g. "checkpoints/pretrain_best.pt" (relative to repo; checkpoints/ is on Drive)

DRIVE_DIR = "/content/drive/MyDrive/mamba-dsr"
PROCESSED_ZIP = f"{DRIVE_DIR}/processed.zip"

# Raw wavs are only used to compute speaker embeddings. Skip when RESUME is a
# checkpoint of the same dataset (embeddings are stored in it).
DOWNLOAD_RAW = True

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive, userdata
drive.mount("/content/drive")

import os
token = userdata.get("GITHUB_TOKEN")
if not os.path.exists("/content/mamba-dsr"):
    !git clone -q https://{token}@github.com/yt-1-dev/mamba-dsr.git /content/mamba-dsr
%cd /content/mamba-dsr
!git log --oneline -1

## Install
Keeps Colab's preinstalled torch and picks the prebuilt mamba-ssm / causal-conv1d wheels that match it
(`requirements.txt` pins a cu130 torch that Colab's driver may not support).

In [ ]:
import sys, torch
tv = ".".join(torch.__version__.split("+")[0].split(".")[:2])
cu = torch.version.cuda.split(".")[0]
py = f"cp{sys.version_info.major}{sys.version_info.minor}"
tag = f"cu{cu}torch{tv}cxx11abiTRUE-{py}-{py}-linux_x86_64"
print("torch", torch.__version__, "->", tag)

CC = f"https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.7.0/causal_conv1d-1.7.0+{tag}.whl"
MS = f"https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1+{tag}.whl"
# torchaudio>=2.8 loads audio through torchcodec, which must match torch
TORCHCODEC = {"2.8": "0.7", "2.9": "0.9", "2.10": "0.10"}.get(tv)
extra = f"torchcodec=={TORCHCODEC}" if TORCHCODEC else ""

!pip install -q {CC} {MS} "speechbrain>=1.0.0" soundfile kaggle {extra}

from mamba_ssm import Mamba3  # fails here, not mid-training, if the wheel doesn't fit
print("mamba_ssm ok")

## Data (preprocessing skipped — uses the `.pt` mels from Drive)

In [ ]:
# DataLoaders.py reads ../data/... relative to the repo, i.e. /content/data
# unzip to local disk; reading ~136k small .pt files straight off Drive is very slow
if not os.path.exists("/content/data/processed"):
    !mkdir -p /content/data && unzip -q "{PROCESSED_ZIP}" -d /content/data
!ls /content/data/processed

if DOWNLOAD_RAW:
    if MODE == "pretrain":
        !sh download_libritts.sh
    else:
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
        !sh download_uaspeech.sh

In [ ]:
# checkpoints/ -> Drive, so train()'s ./checkpoints output persists
os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
if not os.path.islink("checkpoints"):
    !rm -rf checkpoints && ln -s "{DRIVE_DIR}/checkpoints" checkpoints
!ls -la checkpoints/

## Load training code

In [ ]:
from pathlib import Path

# main.py runs set_checkpoint_mode(...) and load_dataloaders('libre', 16) at
# import time (crashes when LibriTTS isn't present). Drop those lines and exec the rest.
# If main.py gains more top-level calls, add their prefixes here.
SKIP = ("set_checkpoint_mode(", "train, test = load_dataloaders", "print(len(train))")
src = "\n".join(l for l in Path("python_files/main.py").read_text().splitlines()
                if not l.startswith(SKIP))
exec(compile(src, "python_files/main.py", "exec"), globals())
print(MODE_CONFIG[MODE])

## Train

In [ ]:
# MODE_CONFIG[MODE]["batch_size"] = 8   # lower if you hit OOM
train(MODE, RESUME, "cuda:0")

Next stage: set `MODE = "finetune"`, `RESUME = "checkpoints/pretrain_best.pt"` in the config cell and re-run
(then `e2e` from `finetune_best.pt`). Checkpoints are in `DRIVE_DIR/checkpoints/`.